In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

In [6]:
import re
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from src.utils.helpers import totalizador
from src.schemas import dispatch
from typing import Optional
from src.schemas.banks.pagcorp import Pagcorp
from src.schemas.parsers.pdf_extractor import PDFExtractor

# origem = r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\00_ABASE\Lançamentos_Contabeis.xls"
# name = "[LANC] - Extrato_890000983290_03-06-2026_Parte1 (1).xls".replace("EXT", "LANC")
# destino = rf"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\EXTRATOS 0626\{name}.xls"
# shutil.copy2(origem, destino)

ext = r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\526_EXTBAN INTER_SILVA.xlsx"

path = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\526_EXTBAN INTER_SILVA.pdf")

if path.suffix == ".xlsx":
    df = pd.read_excel(path)
    df = Pagcorp(df).layout1()
else:
    pdf = PDFExtractor(path).extract()

if pdf:
    df = dispatch(pdf)
    display(pdf)


['Solicitado em: 01/06/2026 - 17h07',
 'Silva E Souza Ltda',
 'CPF/CNPJ: 52.074.420/0001-16, Instituição: Banco Inter, Agência: 0001-9, Conta: 52603659-1',
 'Período: 01/05/2026 a 31/05/2026',
 'Saldo total Saldo disponível: Saldo bloqueado:',
 'R$ 3.130,00 R$ 3.130,00 R$ 0,00',
 '(bloqueado + disponível)',
 '1 de Maio de 2026 Saldo do dia: R$ 1.500,00 Valor Saldo por transação',
 'Pix recebido: "Cp :18236120-Diego Mendes Silva" R$ 1.500,00 R$ 3.366,00',
 'Pix enviado: "Cp :10209619-IMPERIO BOMBAS" -R$ 5.399,00 -R$ 2.033,00',
 'Pix recebido: "Cp :10573521-Rogerio da Silva Pires" R$ 733,00 -R$ 1.300,00',
 'Pix recebido: "Cp :31872495-GRP EMPREENDIMENTOS" R$ 2.800,00 R$ 1.500,00',
 '2 de Maio de 2026 Saldo do dia: R$ 2.800,00',
 'Pix recebido: "Cp :02935307-ESCOLA MAGIA DO SABER FC LTDA" R$ 1.300,00 R$ 2.800,00',
 '4 de Maio de 2026 Saldo do dia: R$ 5.495,35',
 'Pix recebido: "Cp :90400888-EURIPEDES ALVES FERREIRA" R$ 515,00 R$ 3.315,00',
 'Pix recebido: "Cp :71328769-GIOVANNA SERIO" R$ 

In [ ]:
pdf = pdf[7:]
pdf = [
    item for item in pdf
    if not item.startswith(("Fale com a gente", "SAC: "))
]

REGEX_DIA = re.compile(
    r'^(?P<data>\d{1,2}\s+de\s+[A-Za-zçÇãÃéÉíÍóÓúÚ]+\s+de\s+\d{4})'
    r'\s+Saldo do dia:\s+'
    r'(?P<saldo_dia>-?R\$\s*[\d\.]+,\d{2})'
    r'(?:\s+Valor\s+Saldo por transação)?$',
    re.IGNORECASE
)


REGEX_MOVIMENTACAO = re.compile(
    r'^(?P<descricao>.+?)\s+'
    r'(?P<valor>-?R\$\s*[\d\.]+,\d{2})\s+'
    r'(?P<saldo_transacao>-?R\$\s*[\d\.]+,\d{2})$'
)


def brl_para_float(valor: str) -> float:
    """
    Converte valores no formato brasileiro:
    R$ 1.500,00 -> 1500.00
    -R$ 5.399,00 -> -5399.00
    """
    valor = valor.strip()
    valor = valor.replace("R$", "").replace(" ", "")
    valor = valor.replace(".", "").replace(",", ".")

    return float(valor)


def extrair_movimentacoes_inter(linhas: list[str]) -> pd.DataFrame:
    registros = []

    data_atual = None
    saldo_dia_atual = None

    for linha in linhas:
        linha = linha.strip()

        if not linha:
            continue

        # Ignora cabeçalhos/rodapés que não são movimentações
        if linha.startswith(("Solicitado em:", "Fale com a gente", "SAC:", "Ouvidoria:")):
            continue

        match_dia = REGEX_DIA.match(linha)

        if match_dia:
            data_atual = match_dia.group("data")
            saldo_dia_atual = match_dia.group("saldo_dia")
            continue

        match_mov = REGEX_MOVIMENTACAO.match(linha)

        if match_mov and data_atual:
            descricao = match_mov.group("descricao")
            valor = match_mov.group("valor")
            saldo_transacao = match_mov.group("saldo_transacao")

            registros.append({
                "DATA": data_atual,
                "SALDO_DIA": saldo_dia_atual,
                "DESCRICAO": descricao,
                "VALOR_ORIGINAL": valor,
                "VALOR": brl_para_float(valor),
                "SALDO_TRANSACAO_ORIGINAL": saldo_transacao,
                "SALDO_TRANSACAO": brl_para_float(saldo_transacao),
                "TIPO": "C" if not valor.strip().startswith("-") else "D"
            })
    return registros

registros = extrair_movimentacoes_inter(pdf)

display(registros)

None

In [ ]:

import shutil
import pandas as pd
from src.schemas import dispatch
from src.schemas.banks.pagcorp import Pagcorp
from src.schemas.parsers.pdf_extractor import PDFExtractor
from src.utils.helpers import totalizador, planilha_lancamento

fecfin = r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\0426_FECFIN_CEMAF OPE.xlsx"
df_sicoob = pd.read_excel(fecfin, sheet_name="SICOOB 18723-2")
df_sicoob.columns = df_sicoob.iloc[5]
df_sicoob = df_sicoob[6:].reset_index(drop=True)
df_sicoob = df_sicoob.dropna(subset=["DATA"])
df_sicoob = df_sicoob[~df_sicoob["HISTÓRICO"].astype(str).str.upper().str.contains("SALDO", na=False)]
df_sicoob["SAIDA"] = df_sicoob["SAIDA"] * -1
df_sicoob["DESCRIÇÃO"] = df_sicoob.apply(lambda row: f"{row["TIPO"]} {row["Nº DOC"]} {row["HISTÓRICO"]} {row["OBS P/ CONT"]} {row["OBS P/ INTERNAS"]}", axis=1)
df_sicoob["DESCRIÇÃO"] = (df_sicoob["DESCRIÇÃO"].str.strip().str.replace("nan", "").str.replace("  ", " "))
df_sicoob["VALOR"] = df_sicoob.apply(lambda row: row["ENTRADA"] if row["ENTRADA"] > 0 else row["SAIDA"], axis=1)
df_sicoob = df_sicoob[["DATA", "DESCRIÇÃO", "VALOR"]]
df_sicoob["TIPO"] = df_sicoob.apply(lambda x: "C" if x["VALOR"] > 0 else "D", axis=1)

planilha_lancamento(df_sicoob, )
  
display(df_sicoob)


In [ ]:
from src.utils.helpers import totalizador, planilha_lancamento

planilha_lancamento(df, destino)
df = totalizador(df)
df.to_excel(ext, index=False)

display(df)

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
from src.schemas import dispatch
from src.schemas.parsers.pdf_extractor import PDFExtractor
from src.utils.helpers import totalizador, planilha_lancamento


lancamento = Path.cwd() / r"data\Lancamentos_Contabeis.xls"
extratos = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT")

invalidos = Path(extratos / "00_INVALIDOS")
convertidos = Path(extratos / "00_CONVERTIDOS")
invalidos.mkdir(parents=True, exist_ok=True)
convertidos.mkdir(parents=True, exist_ok=True)

if extratos.exists():
    arquivos = [arquivo for arquivo in extratos.iterdir()]
    for arquivo in arquivos:
        if arquivo.is_file() and arquivo.suffix.lower() == ".pdf" and "EXT" in arquivo.stem:
            
            pdf = PDFExtractor(arquivo).extract()

            if not pdf:
                destino = invalidos / arquivo.name
                shutil.move(str(arquivo), str(destino))
                print(f"PDF movido para inválidos porque está vazio: {arquivo.name}")
                continue

            df = dispatch(pdf)

            if df is None or df.empty:
                destino = invalidos / arquivo.name
                shutil.move(str(arquivo), str(destino))
                print(f"PDF movido para inválidos porque o DataFrame veio vazio: {arquivo.name}")
                continue

            # Cria uma pasta dentro de 00_CONVERTIDOS com o nome do arquivo
            pasta_arquivo = convertidos / arquivo.stem
            pasta_arquivo.mkdir(parents=True, exist_ok=True)

            name = f"{arquivo.stem}".replace("EXT", "LANC")

            dest_lancamento = pasta_arquivo / f"{name}.xls"
            dest_excel = pasta_arquivo / f"{arquivo.stem}.xlsx"
            dest_pdf = pasta_arquivo / arquivo.name

            shutil.copy2(lancamento, dest_lancamento)

            planilha_lancamento(df, dest_lancamento)

            df = totalizador(df)

            df.to_excel(dest_excel, index=False)

            shutil.move(str(arquivo), str(dest_pdf))

            print(f"PDF convertido com sucesso: {arquivo.name}")
            print(f"Arquivos salvos em: {pasta_arquivo}")



In [ ]:
from pathlib import Path
from typing import Optional
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

class GoogleDrive():
    def __init__(self, GOOGLE_APPLICATION_CREDENTIALS):
        self.SCOPES = ["https://www.googleapis.com/auth/drive"]
        self.GOOGLE_APPLICATION_CREDENTIALS = GOOGLE_APPLICATION_CREDENTIALS

    def _service(self):
        credentials = Credentials.from_service_account_file(
            self.GOOGLE_APPLICATION_CREDENTIALS,
            scopes=self.SCOPES
        )
        return build("drive", "v3", credentials=credentials)
    
    def list_folder(self, folder_id, folder_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType = '{folder_type}' "
            f"and trashed = false"
        )
        pastas = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageToken = page_token
            ).execute()
            pastas.extend(response.get("files", []))
            page_token = response.get("nextPageToken")

            if not page_token:
                break

        return pastas
    
    def list_folder_by_name(self, folder_id, name_folder, folder_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and name = '{name_folder}' "
            f"and mimeType = '{folder_type}' "
            f"and trashed = false"
        )
        response = service.files().list(
            q=query,
            spaces="drive",
            fields="files(id, name, mimeType)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        pastas = response.get("files", [])
        return pastas[0] if pastas else None

    def list_files(self, folder_id):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType != 'application/vnd.google-apps.folder' "
            f"and trashed = false"
        )

        arquivos = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType, size, modifiedTime)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
                pageToken=page_token
            ).execute()
            arquivos.extend(response.get("files", []))
            page_token = response.get("nextPageToken")
            if not page_token:
                break
        return arquivos
    
    def search_file_by_name(self, folder_id, name_file):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and name = '{name_file}' "
            f"and trashed = false"
        )
        response = service.files().list(
            q=query,
            spaces="drive",
            fields="files(id, name, mimeType, size, modifiedTime)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        arquivos = response.get("files", [])
        return arquivos[0] if arquivos else None
    
    def pdfs(self, folder_id, pdf_type):
        service = self._service()
        query = (
            f"'{folder_id}' in parents "
            f"and mimeType = '{pdf_type}' "
            f"and trashed = false"
        )
        arquivos = []
        page_token = None
        while True:
            response = service.files().list(
                q=query,
                spaces="drive",
                fields="nextPageToken, files(id, name, mimeType, size, modifiedTime)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            arquivos.extend(response.get("files", []))
            page_token = response.get("nextPageToken")
            if not page_token:
                break
        return arquivos
    
    def create_folder(self, name_folder, type_folder, folder_id_pai=None):
        service = self._service()
        metadata = {
            "name": name_folder,
            "mimeType": type_folder
        }

        if folder_id_pai:
            metadata["parents"] = [folder_id_pai]

        pasta = service.files().create(
            body=metadata,
            fields="id, name, mimeType",
            supportsAllDrives=True
        ).execute()
        return pasta
    
    def get_or_create_folder(self, folder_id_pai, name_folder):
        folder_type = "application/vnd.google-apps.folder"

        pasta = self.list_folder_by_name(
            folder_id=folder_id_pai,
            name_folder=name_folder,
            folder_type=folder_type
        )

        if pasta:
            return pasta

        return self.create_folder(
            name_folder=name_folder,
            type_folder=folder_type,
            folder_id_pai=folder_id_pai
        )
    
    def move_file(self, file_id, folder_id_destino):
        service = self._service()

        arquivo = service.files().get(
            fileId=file_id,
            fields="parents",
            supportsAllDrives=True
        ).execute()

        parents_atuais = ",".join(arquivo.get("parents", []))

        arquivo_movido = service.files().update(
            fileId=file_id,
            addParents=folder_id_destino,
            removeParents=parents_atuais,
            fields="id, name, parents",
            supportsAllDrives=True
        ).execute()

        return arquivo_movido
    
    def download(self, file_id, destino_local):
        service = self._service()
        destino = Path(destino_local)
        destino.parent.mkdir(parents=True, exist_ok=True)

        request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
        with open(destino, "wb") as arquivo_local:
            downloader = MediaIoBaseDownload(arquivo_local, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        return str(destino)
    
    def upload(self, caminho_local, folder_id_destino, type_file, name_drive: Optional[str] = None):
        service = self._service()
        caminho = Path(caminho_local)
        if not caminho.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho_local}")
        
        nome_final = name_drive or caminho.name
        metadata = {
            "name": nome_final,
            "parents": [folder_id_destino]
        }
        media = MediaFileUpload(
            filename=str(caminho),
            mimetype=type_file,
            resumable=True
        )

        arquivo = service.files().create(
            body=metadata,
            media_body=media,
            fields="id, name, mimeType, size, modifiedTime",
            supportsAllDrives=True
        ).execute()
        return arquivo
    
